# ShapeNet Dataset

In [1]:
%load_ext autoreload
%autoreload 2

In [48]:
from pathlib import Path
import shutil
import zipfile
import json
import numpy as np
from torch_pointcloud.datasets.shapenet import ShapeNet
import torch_pointcloud.transforms as T

from torch_pointcloud.datasets.utils import download_file, extract_zip

### Download


```python
def download(self) -> None:
    path = download_url(self.url, self.root)
    extract_zip(path, self.root)
    os.unlink(path)
    fs.rm(self.raw_dir)
    name = self.url.split("/")[-1].split(".")[0]
    os.rename(osp.join(self.root, name), self.raw_dir)
```

In [ ]:
import urllib.parse
import os
from urllib.parse import urlparse

url = "http://photographs.500px.com/kyle/09-09-201315-47-571378756077.jpg"
a = urlparse(url)
print(a.path)                    # Output: /kyle/09-09-201315-47-571378756077.jpg
print(os.path.basename(a.path))  # Output: 09-09-201315-47-571378756077.jpg

In [25]:
# url = "https://shapenet.cs.stanford.edu/media/shapenetcore_partanno_segmentation_benchmark_v0.zip"
url = "https://shapenet.cs.stanford.edu/media/shapenetcore_partanno_segmentation_benchmark_v0_normal.zip"
zip_path = "data/shapenet/shapenetcore_partanno_segmentation_benchmark_v0_normal.zip"


path = download_file(zip_path, url)
print(f"{path = }")

Downloading: 100%|██████████| 674M/674M [01:10<00:00, 9.99MB/s]  

path = 'data/shapenet/shapenetcore_partanno_segmentation_benchmark_v0_normal.zip'


In [26]:
data_dir = "data/shapenet"
raw_dir = "data/shapenet/raw"
zip_name = "shapenetcore_partanno_segmentation_benchmark_v0_normal"


shutil.rmtree(raw_dir, ignore_errors=True)
extract_path = extract_zip(path, data_dir)
Path(data_dir, zip_name).rename(raw_dir)

Extracting: 100%|██████████| 16868/16868 [00:08<00:00, 1951.82it/s]


PosixPath('data/shapenet/raw')

In [27]:
# zip_path = "data/shapenet/shapenetcore_partanno_segmentation_benchmark_v0.zip"
# data_dir = "data/shapenet"


# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(data_dir)

### Process


```python
def process_filenames(self, filenames: List[str]) -> List[Data]:
    data_list = []
    categories_ids = [self.category_ids[cat] for cat in self.categories]
    cat_idx = {categories_ids[i]: i for i in range(len(categories_ids))}

    for name in filenames:
        cat = name.split(osp.sep)[0]
        if cat not in categories_ids:
            continue

        tensor = read_txt_array(osp.join(self.raw_dir, name))
        pos = tensor[:, :3]
        x = tensor[:, 3:6]
        y = tensor[:, -1].type(torch.long)
        data = Data(pos=pos, x=x, y=y, category=cat_idx[cat])
        if self.pre_filter is not None and not self.pre_filter(data):
            continue
        if self.pre_transform is not None:
            data = self.pre_transform(data)
        data_list.append(data)

    return data_list
```

In [ ]:

extract_path

'data/shapenet'

In [32]:
CATEGORY_IDS = {
    "Airplane": "02691156",
    "Bag": "02773838",
    "Cap": "02954340",
    "Car": "02958343",
    "Chair": "03001627",
    "Earphone": "03261776",
    "Guitar": "03467517",
    "Knife": "03624134",
    "Lamp": "03636649",
    "Laptop": "03642806",
    "Motorbike": "03790512",
    "Mug": "03797390",
    "Pistol": "03948459",
    "Rocket": "04099429",
    "Skateboard": "04225987",
    "Table": "04379243",
}

SEG_CLASSES = {
    "Airplane": [0, 1, 2, 3],
    "Bag": [4, 5],
    "Cap": [6, 7],
    "Car": [8, 9, 10, 11],
    "Chair": [12, 13, 14, 15],
    "Earphone": [16, 17, 18],
    "Guitar": [19, 20, 21],
    "Knife": [22, 23],
    "Lamp": [24, 25, 26, 27],
    "Laptop": [28, 29],
    "Motorbike": [30, 31, 32, 33, 34, 35],
    "Mug": [36, 37],
    "Pistol": [38, 39, 40],
    "Rocket": [41, 42, 43],
    "Skateboard": [44, 45, 46],
    "Table": [47, 48, 49],
}

In [40]:
categories = ["Airplane"]
split = "train"

In [46]:
data_list = []
# categories_ids = [CATEGORY_IDS[cat] for cat in categories]
# cat_idx = {categories_ids[i]: i for i in range(len(categories_ids))}
category_id_to_idx = {CATEGORY_IDS[cat]: i for i, cat in enumerate(categories)}
print(f"{category_id_to_idx = }")

category_id_to_idx = {'02691156': 0}


In [64]:
import torch


split_path = Path(raw_dir, "train_test_split", f"shuffled_{split}_file_list.json")

with open(split_path, "r") as f:
    split_data = json.load(f)
    
    
data_list = []
for file_name in split_data:
    file_path = Path(raw_dir, file_name.replace("shape_data/", "")).with_suffix(".txt")
    category_id = file_path.parent.name
    if category_id not in category_id_to_idx:
        continue
    
    # Process file
    data = np.loadtxt(file_path, delimiter=" ")
    xyz = data[:, :3]
    normal = data[:, 3:6]
    seg_target = data[:, -1]
    
    data = {
        "xyz": torch.from_numpy(xyz).float(),
        "normal": torch.from_numpy(normal).float(),
        "seg_target": torch.from_numpy(seg_target).long(),
        "cat_target": category_id_to_idx[category_id],
    }
    # End process file

    # if self.pre_filter is not None and not self.pre_filter(data):
    #     continue
    # if self.pre_transform is not None:
    #     data = self.pre_transform(data)
    
    data_list.append(data)

print(f"{len(data_list) = :,}")

len(data_list) = 1,958


In [65]:
data

{'xyz': tensor([[ 0.2405, -0.0107, -0.0213],
         [ 0.3266, -0.0688, -0.0186],
         [ 0.1758, -0.0246, -0.0313],
         ...,
         [-0.0614, -0.0435,  0.1980],
         [ 0.0333, -0.0731, -0.0838],
         [-0.3167,  0.0010,  0.0016]]),
 'normal': tensor([[ 0.0068,  0.6594, -0.7518],
         [ 0.1610, -0.6245, -0.7642],
         [ 0.0085,  0.2301, -0.9731],
         ...,
         [-0.0037,  0.9919, -0.1266],
         [-0.3770,  0.2891,  0.8799],
         [-0.0653,  0.8816,  0.4674]]),
 'seg_target': tensor([0, 0, 0,  ..., 1, 3, 2]),
 'cat_target': 0}

In [ ]:
data_list = []
categories_ids = [self.category_ids[cat] for cat in self.categories]
cat_idx = {categories_ids[i]: i for i in range(len(categories_ids))}

for name in filenames:
    cat = name.split(osp.sep)[0]
    if cat not in categories_ids:
        continue

    tensor = read_txt_array(osp.join(self.raw_dir, name))
    pos = tensor[:, :3]
    x = tensor[:, 3:6]
    y = tensor[:, -1].type(torch.long)
    data = Data(pos=pos, x=x, y=y, category=cat_idx[cat])
    if self.pre_filter is not None and not self.pre_filter(data):
        continue
    if self.pre_transform is not None:
        data = self.pre_transform(data)
    data_list.append(data)